## EDA - Energy Demand vs Weather


### Load weather data from open-meteo with API request

* over open-meteo archive API
* fetch weather data of the selected cities from 2019-01-01 till 2025-09-30 (time range of the electricity load file)
* get the latitude and longitude of the selected cities
* construct the request with header information with paramaters including the following

- latitude and longitude of the selected cities
	*	Berlin, Hamburg, München, Köln, Frankfurt
- start and end date time, hourly, the weather variables to be requested 
	* `temperature_2m`
	* `apparent_temperature`
	* `precipitation`
	* `rain`
	* `snowfall`
	* `wind_speed_10m`
	* `shortwave_radiation`

* save the dataframes to csv for later usage

### variables to be requested
* apparent_temperature 
    - perceived temperature, which takes into account factors such as humidity and wind speed to provide a more accurate representation of how the temperature feels to humans. It is calculated using a formula that combines the actual air temperature with the effects of humidity and wind chill. The apparent temperature can be higher than the actual temperature in hot and humid conditions, and lower than the actual temperature in cold and windy conditions.
* precipitation 
    - the amount of water that falls from the atmosphere to the ground in the form of rain, snow, sleet, or hail. It is typically measured in millimeters (mm) or inches (in) and can be used to assess the amount of moisture in the air and the likelihood of certain weather conditions, such as flooding or drought.
* shortwave_radiation
    - the amount of solar radiation that reaches the Earth's surface in the form of shortwave electromagnetic waves. It is typically measured in watts per square meter (W/m²) and can be used to assess the amount of energy available for photosynthesis, as well as the potential for solar power generation.
* weathercode 
    - a numerical code that represents the current weather conditions. It is typically used in weather forecasting and can be used to quickly identify the type of weather that is expected, such as clear skies, rain, snow, or thunderstorms.

In [ ]:
import time
import requests
import pandas as pd 

weather_variables = ['temperature_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'shortwave_radiation', 'weathercode']

# get latitude and longitude of German cities: Berlin, Hamburg, München, Köln, Frankfurt
selected_cities = {  
    'Berlin': {'latitude': 52.5200, 'longitude': 13.4050},
    'Hamburg': {'latitude': 53.5511, 'longitude': 9.9937},
    'München': {'latitude': 48.1351, 'longitude': 11.5820},
    'Köln': {'latitude': 50.9375, 'longitude': 6.9603},
    'Frankfurt': {'latitude': 50.1109, 'longitude': 8.6821}
}

start_date = "2019-01-01"
end_date = "2025-09-30"
for city, coords in selected_cities.items():
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={coords['latitude']}&longitude={coords['longitude']}&start_date={start_date}&end_date={end_date}&hourly={','.join(weather_variables)}&timezone=auto"
    response = requests.get(url)
    weather_data = response.json()
    df_weather_city = pd.DataFrame(weather_data['hourly'])
    df_weather_city['time'] = pd.to_datetime(df_weather_city['time'], utc=True)
    #display(df_weather_city.head(3))
    
    # save the weather data for each city to a separate csv file in case of API issues in the future
    df_weather_city.to_csv(f"../data/tmp/weather_data_{city}.csv", index=False)
    print(f"Weather data loaded and saved for: {city}")
    time.sleep(2)  # Sleep for 2 seconds to avoid hitting API rate limits

print("Weather data collection completed for all selected cities and years.")


In [ ]:
# calculate the weight of the cities based on their population size and use it to create a weighted average of the weather variables for Germany
import pandas as pd 

weather_variables = ['temperature_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'shortwave_radiation', 'weathercode']

city_population = {
    'Berlin': 3644826, 
    'Hamburg': 1841179, 
    'München': 1471508,     
    'Köln': 1085664, 
    'Frankfurt': 753056 
}

total_population = sum(city_population.values())
city_weights = {city: population / total_population for city, population in city_population.items()}
df_weather_germany = pd.DataFrame()
for city, weight in city_weights.items():
    df_city = pd.read_csv(f"../data/tmp/weather_data_{city}.csv")
    df_city_weighted = df_city.copy()
    for var in weather_variables:
        df_city_weighted[var] *= weight
    if df_weather_germany.empty:
        df_weather_germany = df_city_weighted
    else:
        df_weather_germany[weather_variables] += df_city_weighted[weather_variables]
df_weather_germany.to_csv("../data/processed/weather_data_de_2019_2025.csv", index=False)
df_weather_germany.head()


In [ ]:
# check outliers in the weather data using boxplots for each weather variable
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 4, figsize=(16, 8))
ax = ax.flatten()

for i, variable in enumerate(weather_variables): 
    ax[i].boxplot(df_weather_germany[variable])
    ax[i].set_title(f'Boxplot for {variable}')
    ax[i].set_ylabel(variable)
fig.tight_layout()
plt.show()

In [ ]:
# drop the weathercode column as it is not a continuous variable and cannot be used for regression analysis
df_weather_germany = df_weather_germany.drop('weathercode', axis=1)

In [ ]:
import seaborn as sns

weather_corr = df_weather_germany.drop('time', axis=1).corr()
sns.heatmap(weather_corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap of Weather Variables') 


In [ ]:
# drop the precipitation and temperature_2m columns as they are highly correlated with rain and apparent_temperature, respectively (see correlation heatmap in notebook 02 EDA)
df_weather_germany = df_weather_germany.drop(columns=['precipitation', 'temperature_2m'], axis=1)
df_weather_germany = df_weather_germany.rename(columns={'time':"DateUTC"})

In [ ]:
df_weather_germany.to_csv("../data/weather_data_mean_cities_2019_2025.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


df_weather_germany = pd.read_csv("../data/processed/weather_data_mean_cities_2019_2025.csv", parse_dates=['DateUTC'])

# resample weather data to monthly mean values
df_weather_monthly = df_weather_germany.set_index('DateUTC').resample('ME').mean().reset_index()

# plot monthly mean values of weather variables over time
fig, ax = plt.subplots(2, 3, figsize=(18, 5))  
ax = ax.flatten()
weather_variables_monthly = ['apparent_temperature', 'rain', 'snowfall', 'wind_speed_10m', 'shortwave_radiation']
for i, variable in enumerate(weather_variables_monthly):  
    sns.lineplot(data=df_weather_monthly, x='DateUTC', y=variable, ax=ax[i])
    ax[i].set_title(f'{variable} over Time')
    ax[i].set_xlabel('Date')
    ax[i].set_ylabel(variable)
    ax[i].grid()

ax[len(weather_variables_monthly)].axis('off')
plt.tight_layout()
plt.show()
